# 📘 Módulo 06: Agregación, Agrupamiento (GROUP BY, HAVING) & Subconsultas
### 📊 Guía Ultramétrica y Dócil: Paso a Paso, Diagramas E-R, Regla de Oro del GROUP BY, WHERE vs HAVING, Subconsultas Escalares, de Lista (IN/NOT IN), Tabla Derivada (FROM) y Correlacionadas (EXISTS)
**Asignatura:** Bases de Datos Avanzadas  
**Entorno:** Google Colab / DuckDB (100% In-Memory / Zero 404 Error)  

---
## 🎯 Objetivos de Aprendizaje
1. **Comprender la lógica de agrupamiento bidimensional y tridimensional:** Dominar `GROUP BY` con 1, 2 o más columnas y aplicar las 5 funciones de agregación (`COUNT`, `SUM`, `AVG`, `MAX`, `MIN`).
2. **Diferenciar estrictamente WHERE vs HAVING:** Saber exactamente cuándo filtrar filas individuales (pre-agregación) y cuándo filtrar resúmenes de grupos (post-agregación).
3. **Dominar la Metodología Paso a Paso de Subconsultas:** Construir subconsultas escalares (`WHERE val > (SELECT...)`), en el `SELECT` (columnas calculadas), de lista (`IN`, `NOT IN`) y tablas derivadas (`FROM (SELECT...)`).
4. **Comprender la trampa del NULL en `NOT IN` vs `NOT EXISTS`:** Evitar errores clásicos de producción al comparar conjuntos con nulos.
5. **Desarrollar Subconsultas Correlacionadas:** Analizar la evaluación fila a fila para comparar registros contra el promedio o máximo de su propia categoría.

In [ ]:
# ==============================================================================
# 🚀 SETUP INICIAL AUTO-CONTENIDO: DUCKDB & GOOGLE COLAB (100% IN-MEMORY)
# ==============================================================================
!pip install duckdb duckdb-engine jupysql -q

import duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# 1. Conectar a motor DuckDB en memoria
conn = duckdb.connect(database=':memory:')

# 2. Poblar esquema de datos enriquecido de la Empresa CUN
conn.execute("""
CREATE TABLE IF NOT EXISTS departamentos (
    depto_id INT PRIMARY KEY,
    nombre VARCHAR,
    presupuesto DECIMAL(15,2),
    ciudad VARCHAR
);
INSERT INTO departamentos VALUES
(1, 'Tecnología e Innovación', 150000000.00, 'Bogotá'),
(2, 'Ventas y Comercio', 200000000.00, 'Medellín'),
(3, 'Operaciones y Logística', 120000000.00, 'Cali'),
(4, 'Recursos Humanos', 80000000.00, 'Bogotá'),
(5, 'Finanzas y Contabilidad', 110000000.00, 'Barranquilla');

CREATE TABLE IF NOT EXISTS empleados (
    empleado_id INT PRIMARY KEY,
    nombre VARCHAR,
    salario DECIMAL(15,2),
    fecha_ingreso DATE,
    jefe_id INT,
    depto_id INT REFERENCES departamentos(depto_id)
);
INSERT INTO empleados VALUES
(101, 'Carlos Mendoza', 8500000.00, '2021-03-15', NULL, 1),
(102, 'Ana María Gómez', 6200000.00, '2022-01-10', 101, 1),
(103, 'Luis Fernando Torres', 4800000.00, '2023-05-20', 101, 1),
(104, 'Sofía Ramírez', 9000000.00, '2020-11-01', NULL, 2),
(105, 'Diego Alejandro Silva', 5500000.00, '2022-08-14', 104, 2),
(106, 'Valentina Morales', 5100000.00, '2023-02-28', 104, 2),
(107, 'Javier Castro', 7000000.00, '2019-06-12', NULL, 3),
(108, 'Mariana López', 4200000.00, '2024-01-15', 107, 3),
(109, 'Andrés Felipe Herrera', 6500000.00, '2021-09-01', NULL, 4),
(110, 'Camila Ospina', 5800000.00, '2022-10-05', NULL, 5),
(111, 'Patricia Alarcón', 3900000.00, '2024-03-01', 109, 4); -- Empleado junior sin pedidos

CREATE TABLE IF NOT EXISTS clientes (
    cliente_id INT PRIMARY KEY,
    nombre VARCHAR,
    ciudad VARCHAR,
    categoria_cliente VARCHAR
);
INSERT INTO clientes VALUES
(201, 'Tech Solutions S.A.S.', 'Bogotá', 'Empresarial'),
(202, 'Comercializadora del Valle', 'Cali', 'Empresarial'),
(203, 'Juan Pablo Restrepo', 'Medellín', 'Individual'),
(204, 'Inversiones Caribe Ltd.', 'Barranquilla', 'Empresarial'),
(205, 'Diana Marcela Vargas', 'Bogotá', 'VIP'),
(206, 'Servicios Logísticos del Eje', 'Pereira', 'Empresarial'),
(207, 'Mateo Gutiérrez', 'Bucaramanga', 'Individual');

CREATE TABLE IF NOT EXISTS productos (
    producto_id INT PRIMARY KEY,
    nombre VARCHAR,
    categoria VARCHAR,
    precio DECIMAL(15,2),
    stock INT
);
INSERT INTO productos VALUES
(301, 'Servidor Cloud Enterprise', 'Infraestructura', 12500000.00, 15),
(302, 'Laptop Pro Developer 16', 'Hardware', 8200000.00, 30),
(303, 'Licencia Base de Datos Pro', 'Software', 4500000.00, 100),
(304, 'Switch Administrable 48P', 'Redes', 3100000.00, 25),
(305, 'Monitor Ultrawide 34 IPS', 'Periféricos', 2400000.00, 45),
(306, 'Router Industrial VPN', 'Redes', 1850000.00, 20),
(307, 'Laptop Ultralight 14', 'Hardware', 5400000.00, 40);

CREATE TABLE IF NOT EXISTS pedidos (
    pedido_id INT PRIMARY KEY,
    cliente_id INT REFERENCES clientes(cliente_id),
    empleado_id INT REFERENCES empleados(empleado_id),
    fecha_pedido DATE,
    estado VARCHAR,
    total DECIMAL(15,2)
);
INSERT INTO pedidos VALUES
(501, 201, 104, '2024-01-10', 'Completado', 33200000.00),
(502, 202, 105, '2024-01-15', 'Completado', 18600000.00),
(503, 205, 104, '2024-02-01', 'Completado', 8200000.00),
(504, 203, 106, '2024-02-12', 'Procesando', 2400000.00),
(505, 204, 105, '2024-02-20', 'Completado', 13700000.00),
(506, 201, 104, '2024-03-05', 'Completado', 25000000.00),
(507, 206, 106, '2024-03-18', 'Pendiente', 3100000.00);
""")

%sql conn
print('✅ Entorno DuckDB inicializado y Base de Datos Empresa CUN cargada para Módulo 06.')

## 1. Guía Dócil: ¿Cómo Funciona la Agrupación (`GROUP BY`)?

Cuando usamos `GROUP BY`, la base de datos toma todas las filas individuales de una tabla y las **colapsa** en grupos basados en los valores de una o más columnas. Dentro de cada grupo, podemos calcular estadísticas resumidas usando **funciones de agregación** (`COUNT`, `SUM`, `AVG`, `MAX`, `MIN`).

### 📐 Diagrama E-R: Relación Departamentos - Empleados

```mermaid
erDiagram
    DEPARTAMENTOS ||--o{ EMPLEADOS : contiene
    DEPARTAMENTOS {
        int depto_id PK
        string nombre
        decimal presupuesto
    }
    EMPLEADOS {
        int empleado_id PK
        string nombre
        decimal salario
        int depto_id FK
    }
```

--- 

### 🧠 Las 5 Preguntas Clave para Plantear un `GROUP BY`

1. **¿Qué entidad estoy agrupando?**  
   *Respuesta:* La tabla o combinación de tablas del `FROM`.
2. **¿Por qué atributo(s) quiero resumir?**  
   *Respuesta:* Coloque esa columna en el `GROUP BY` y en el `SELECT`.
3. **¿Qué métrica quiero calcular sobre cada grupo?**  
   *Respuesta:* `COUNT(*)` para contar filas, `SUM(monto)` para sumar valores, `AVG(salario)` para el promedio, etc.
4. **¿La columna del SELECT está en el GROUP BY?**  
   *Respuesta:* **REGLA DE ORO:** Si una columna no está envuelta en `SUM/AVG/COUNT`, **DEBE** estar en el `GROUP BY`.
5. **¿Necesito ordenar el resumen final?**  
   *Respuesta:* Use `ORDER BY` sobre la columna agregada (ej. `ORDER BY nomina_total DESC`).

### 💡 Ejemplo Didáctico 1.1: Agrupamiento por 1 Columna (Nómina por Departamento)

In [ ]:
%%sql
-- Agrupamiento por 1 columna (d.nombre)
SELECT d.nombre AS departamento,
       COUNT(e.empleado_id) AS total_empleados,
       SUM(e.salario) AS nomina_total,
       AVG(e.salario) AS salario_promedio
FROM   departamentos d
JOIN   empleados e ON d.depto_id = e.depto_id
GROUP BY d.nombre
ORDER BY nomina_total DESC;

### 💡 Ejemplo Didáctico 1.2: Agrupamiento Bidimensional (2 Columnas: Categoría de Producto y Stock)

In [ ]:
%%sql
-- Queremos saber cuántos productos hay y cuál es el valor de inventario por CATEGORÍA
SELECT categoria,
       COUNT(producto_id) AS variedades,
       SUM(stock) AS stock_total_unidades,
       AVG(precio) AS precio_promedio,
       SUM(precio * stock) AS valor_inventario_total
FROM   productos
GROUP BY categoria
ORDER BY valor_inventario_total DESC;

## 2. El Experimento Mental: `WHERE` vs `HAVING`

Muchos estudiantes se confunden entre `WHERE` y `HAVING`. La regla mnemotécnica es muy sencilla:

- **`WHERE` actúa antes de agrupar:** Filtra **filas individuales** de la tabla original antes de que el motor empiece a hacer los cálculos de suma o promedio.
- **`HAVING` actúa después de agrupar:** Filtra **resúmenes de grupos** una vez que las funciones de agregación (`SUM`, `AVG`, `COUNT`) ya fueron calculadas.

```mermaid
flowchart LR
    T["Tabla Original"] --> W["WHERE: Filtra Filas Individuales"]
    W --> G["GROUP BY: Colapsa en Grupos"]
    G --> H["HAVING: Filtra Grupos Resumidos"]
    H --> S["SELECT / ORDER BY: Resultado Final"]
```

### 💡 Ejemplo Didáctico 2.1: `WHERE` y `HAVING` Juntos en la Misma Consulta
**Pregunta de Negocio:** Queremos analizar el total de ventas acumuladas por cliente, pero:
1. Solo considerando pedidos que ya estén en estado **'Completado'** (`WHERE estado = 'Completado'`).
2. Y mostrando únicamente a los clientes que hayan acumulado compras totales **superiores a $20,000,000** (`HAVING SUM(total) > 20000000`).

In [ ]:
%%sql
SELECT c.nombre AS cliente,
       COUNT(p.pedido_id) AS compras_completadas,
       SUM(p.total) AS total_facturado
FROM   clientes c
JOIN   pedidos p ON c.cliente_id = p.cliente_id
WHERE  p.estado = 'Completado'           -- 1. Filtro PRE-AGRUPADO (Descarta pedidos pendientes/procesando)
GROUP BY c.nombre                       -- 2. Agrupa por cliente
HAVING SUM(p.total) > 20000000.00       -- 3. Filtro POST-AGRUPADO (Conserva solo clientes con facturación > 20M)
ORDER BY total_facturado DESC;

## 3. Método Paso a Paso para Desmitificar Subconsultas (Subqueries)

Las subconsultas asustan al principio porque vemos código dentro de código. Pero la clave es abordarlas de **adentro hacia afuera** (*Inside-Out*).

### 📐 Diagrama E-R: Clientes $\rightarrow$ Pedidos $\rightarrow$ Empleados

```mermaid
erDiagram
    CLIENTES ||--o{ PEDIDOS : realiza
    EMPLEADOS ||--o{ PEDIDOS : atiende
    CLIENTES {
        int cliente_id PK
        string nombre
        string categoria_cliente
    }
    PEDIDOS {
        int pedido_id PK
        int cliente_id FK
        int empleado_id FK
        decimal total
    }
    EMPLEADOS {
        int empleado_id PK
        string nombre
        decimal salario
    }
```

--- 

### 🛠️ Las 4 Reglas de la Metodología Paso a Paso

#### **Paso 1: Aislar y probar la Subconsulta Interna (*Inner Query*)**
Escriba únicamente la consulta pequeña que responde la duda intermedia. Ejecútela en Colab.
- Si devuelve `1 valor` (ej. `6560000.00`), use operadores escalares (`=`, `>`, `<`).
- Si devuelve `1 columna con varias filas` (ej. `[201, 202, 205]`), use `IN` o `NOT IN`.

#### **Paso 2: Escribir la Consulta Externa (*Outer Query*) dejando un espacio en blanco**
Escriba la estructura principal: `SELECT ... FROM ... WHERE columna OP (...)`.

#### **Paso 3: Envolver la Subconsulta entre Paréntesis `(...)` e Insertarla**
Pegue la subconsulta verificada dentro de los paréntesis.

#### **Paso 4: Validar Coherencia de Tipos de Datos**
Si la consulta externa compara `cliente_id = (...)`, asegúrese de que la subconsulta interna seleccione exactamente `cliente_id` (mismo tipo de dato).

## 4. Los 4 Tipos Principales de Subconsultas Explicados

### 4.1 Subconsulta Escalar en `WHERE` (Compara contra 1 solo valor)
**Pregunta:** Queremos ver todos los empleados que ganan **más que el salario promedio** de la compañía.

In [ ]:
%%sql
-- PASO 1 (Probar subconsulta interna): Devuelve 6560000.00
-- SELECT AVG(salario) FROM empleados;

-- PASO 2 y 3 (Unir con consulta externa):
SELECT empleado_id, nombre, salario
FROM   empleados
WHERE  salario > (
    SELECT AVG(salario) 
    FROM   empleados
)
ORDER BY salario DESC;

### 4.2 Subconsulta Escalar en el `SELECT` (Columna Calculada)
**Pregunta:** Queremos mostrar una lista de todos los productos, su precio, y al lado la **diferencia en pesos** con respecto al producto más caro de la tienda.

In [ ]:
%%sql
SELECT nombre, 
       precio,
       (SELECT MAX(precio) FROM productos) AS precio_maximo_tienda,
       ((SELECT MAX(precio) FROM productos) - precio) AS diferencia_vs_maximo
FROM   productos
ORDER BY precio DESC;

### 4.3 Subconsulta de Lista (`IN` / `NOT IN`)
**Pregunta:** ¿Cuáles clientes **SÍ** han realizado pedidos mayores a $10,000,000?

In [ ]:
%%sql
-- PASO 1: Subconsulta genera lista de cliente_id = [201, 202, 204]
-- PASO 2: La consulta externa busca clientes cuyo ID esté en esa lista
SELECT cliente_id, nombre, ciudad, categoria_cliente
FROM   clientes
WHERE  cliente_id IN (
    SELECT cliente_id 
    FROM   pedidos 
    WHERE  total > 10000000.00
);

### ⚠️ 4.4 Cuidado con la Trampa de `NOT IN` vs `NOT EXISTS` (El Caso de los Valores Nulos)
Si intenta buscar *"Empleados que NUNCA han atendido un pedido"* usando `NOT IN`:
- Si la subconsulta interna devuelve accidentalmente un valor `NULL`, **toda la consulta `NOT IN` fallará retornando 0 filas** debido a la lógica trivalente de SQL (`TRUE AND UNKNOWN = UNKNOWN`).
- **Solución Enterprise:** Use `NOT EXISTS` o `LEFT JOIN ... WHERE key IS NULL`.

In [ ]:
%%sql
-- Buscar empleados que NO han procesado ningún pedido de manera segura usando NOT EXISTS
SELECT e.empleado_id, e.nombre, e.salario
FROM   empleados e
WHERE  NOT EXISTS (
    SELECT 1 
    FROM   pedidos p 
    WHERE  p.empleado_id = e.empleado_id  -- Correlación Fila a Fila
);

### 4.5 Subconsulta en la Cláusula `FROM` (Tabla Derivada / Inline View)
Podemos tratar el resultado de una subconsulta como si fuera una **tabla temporal con nombre propio** dentro del `FROM`.

In [ ]:
%%sql
-- Primero creamos una tabla derivada llamada 'resumen_deptos' y luego calculamos el promedio de las nóminas
SELECT AVG(resumen_deptos.nomina_total) AS nomina_promedio_entre_departamentos
FROM (
    SELECT depto_id, SUM(salario) AS nomina_total
    FROM   empleados
    GROUP BY depto_id
) AS resumen_deptos;

## 5. Subconsultas Correlacionadas Avanzadas

Una subconsulta es **correlacionada** cuando depende de los valores de la fila actual de la consulta externa. El motor ejecuta la subconsulta interna *una vez por cada fila* examinada.

### 💡 Ejemplo Didáctico 5.1: Encontrar Productos que Cuestan Más que el Promedio de su Propia Categoría
**Escenario:** No queremos comparar un portátil con un router. Queremos comparar cada producto **exclusivamente contra los productos de su misma categoría**.

In [ ]:
%%sql
SELECT p.producto_id, p.nombre, p.categoria, p.precio
FROM   productos p
WHERE  p.precio > (
    SELECT AVG(sub.precio)
    FROM   productos sub
    WHERE  sub.categoria = p.categoria  -- Correlación: Conecta la categoría externa con la interna
)
ORDER BY p.categoria, p.precio DESC;

## 6. Retos Prácticos de Negocio con Solución Guiada

### 🧪 Reto 1: Departamentos con Ejecución de Nómina Crítica
**Enunciado:** Encuentre los departamentos cuyo presupuesto total es menor a $120,000,000 y que además tengan al menos 2 empleados contratados.

In [ ]:
%%sql
SELECT d.nombre AS departamento,
       d.presupuesto,
       COUNT(e.empleado_id) AS total_empleados,
       SUM(e.salario) AS nomina_actual
FROM   departamentos d
JOIN   empleados e ON d.depto_id = e.depto_id
WHERE  d.presupuesto < 120000000.00              -- Filtro de Filas
GROUP BY d.nombre, d.presupuesto                -- Agrupamiento
HAVING COUNT(e.empleado_id) >= 2               -- Filtro de Grupo
ORDER BY nomina_actual DESC;

### 🧪 Reto 2: El Empleado de Mayor Salario por Cada Departamento (Subconsulta Correlacionada)
**Enunciado:** Muestre el empleado que ostenta el salario más alto dentro de su respectivo departamento.

In [ ]:
%%sql
SELECT e.empleado_id, e.nombre, e.salario, d.nombre AS departamento
FROM   empleados e
JOIN   departamentos d ON e.depto_id = d.depto_id
WHERE  e.salario = (
    SELECT MAX(sub.salario)
    FROM   empleados sub
    WHERE  sub.depto_id = e.depto_id  -- Correlación Fila por Fila
)
ORDER BY e.salario DESC;

## 📌 Resumen de Competencias Adquiridas en el Módulo 06

- [x] **Agrupamiento Relacional (`GROUP BY`):** Comprensión profunda de agrupaciones por 1 y 2 columnas y cumplimiento de la Regla de Oro.
- [x] **Distinción `WHERE` vs `HAVING`:** Dominio del flujo mnemotécnico (*Filas de entrada $\rightarrow$ Grupos $\rightarrow$ Resúmenes*).
- [x] **Metodología de Subconsultas:** Construcción paso a paso de subconsultas escalares en `WHERE` y `SELECT`, listas `IN`, tablas derivadas en `FROM` y `NOT EXISTS`.
- [x] **Subconsultas Correlacionadas:** Comparación de registros contra el promedio/máximo de su propia categoría.
- [x] **Garantía In-Memory:** Ejecución 100% interactiva en Google Colab con DuckDB sin dependencias externas (Zero 404 Error).